In [1]:
from math import pi
import numpy as np
import scipy as sp
from qiskit.quantum_info import Pauli
from qiskit.circuit.library import PauliEvolutionGate
from qiskit import QuantumCircuit
from qiskit_aer import Aer
from qiskit.circuit import Parameter
from qiskit.circuit.library import TwoLocal
from qiskit_algorithms.optimizers import SciPyOptimizer
from qiskit.primitives import Sampler # new
from qiskit.quantum_info import SparsePauliOp,PauliList
from qiskit.circuit import QuantumCircuit, QuantumRegister
from qiskit.primitives import Estimator # new
import time
import pandas as pd
from qiskit.providers.basic_provider import BasicProvider #new
from multiprocessing import Pool
import multiprocessing as mp
from qiskit_algorithms.optimizers import SciPyOptimizer
import scipy as sp
import os
#from sko.SA import SA

In [2]:
def create_ham_str(nqubits):

    # Create a list of terms for the hamiltonian (open boundary conditions)
    # Inputs: nqubits (int), number of qubits
    # Outputs: ham (list), hamiltonian 

    ham = []

    for i in range(nqubits-1):

        term = ''

        for j in range(nqubits-i-2):

            term += 'I'

        for j in range(nqubits-i-2,nqubits-i):

            term += 'Y'  # Choose a Pauli matrix, i.e., X, Y or Z

        for j in range(nqubits-i,nqubits):

            term += 'I'

        ham.append(term)

    return ham



def evaluate_expectation(parameters_values):
    value_dict = dict(zip(ansatz.parameters, parameters_values))
    pars = list(value_dict.values())
    expectation_value = estimator.run(ansatz, qubit_op, pars).result().values
    return np.real(expectation_value[0])  # Ensure it returns a scalar

In [14]:
############################################################################

# Created by: Prof. Valdecy Pereira, D.Sc.
# UFF - Universidade Federal Fluminense (Brazil)
# email:  valdecy.pereira@gmail.com
# Metaheuristic: Arithmetic Optimization Algorithm

# PEREIRA, V. (2022). GitHub repository: https://github.com/Valdecy/pyMetaheuristic

############################################################################

# Required Libraries
import numpy  as np

############################################################################

# Function
def target_function():
    return

############################################################################

# Function: Initialize Variables
def initial_variables(size, min_values, max_values, target_function, start_init = None):
    dim = len(min_values)
    if (start_init is not None):
        start_init = np.atleast_2d(start_init)
        n_rows     = size - start_init.shape[0]
        if (n_rows > 0):
            rows       = np.random.uniform(min_values, max_values, (n_rows, dim))
            start_init = np.vstack((start_init[:, :dim], rows))
        else:
            start_init = start_init[:size, :dim]
        fitness_values = target_function(start_init) if hasattr(target_function, 'vectorized') else np.apply_along_axis(target_function, 1, start_init)
        population     = np.hstack((start_init, fitness_values[:, np.newaxis] if not hasattr(target_function, 'vectorized') else fitness_values))
    else:
        population     = np.random.uniform(min_values, max_values, (size, dim))
        fitness_values = target_function(population) if hasattr(target_function, 'vectorized') else np.apply_along_axis(target_function, 1, population)
        population     = np.hstack((population, fitness_values[:, np.newaxis] if not hasattr(target_function, 'vectorized') else fitness_values))
    return population

############################################################################

# Function: Update Population
def update_population(population, elite, mu, moa, mop, min_values, max_values, target_function):
    e        = 2.2204e-16
    dim      = len(min_values)
    p        = np.copy(population)
    r1       = np.random.rand(population.shape[0], dim)
    r2       = np.random.rand(population.shape[0], dim)
    r3       = np.random.rand(population.shape[0], dim)
    update_1 = np.where(r1 > moa, elite[:-1] / (mop + e) * ((max_values - min_values) * mu + min_values), elite[:-1])
    update_2 = np.where(r2 <= 0.5, update_1 * mop, update_1 - mop)
    update_3 = np.where(r3  > 0.5, update_2 - ((max_values - min_values) * mu + min_values), update_2 + ((max_values - min_values) * mu + min_values))
    up_pos   = np.clip(update_3, min_values, max_values)
    for i in range(population.shape[0]):
        new_fitness = target_function(up_pos[i, :])
        if (new_fitness < population[i, -1]):
            p[i, :-1] = up_pos[i, :]
            p[i,  -1] = new_fitness
    return p

############################################################################

# Function: AOA
def arithmetic_optimization_algorithm(size = 250, min_values = [-5,-5], max_values = [5,5], iterations = 1500, alpha = 0.5, mu = 5, target_function = target_function, verbose = True, start_init = None, target_value = None, qubits = 3):    
    population = initial_variables(size, min_values, max_values, target_function, start_init)
    elite      = np.copy(population[population[:,-1].argsort()][0,:]) 
    min_values = np.array(min_values)
    max_values = np.array(max_values)
    count      = 0  
    while (count <= iterations and qubits - elite[-1] -1 > 1e-1): 
        if (verbose == True):    
            print('Iteration = ', count, ' f(x) = ', elite[-1]) 
        moa        = 0.2 + count*((1 - 0.2)/iterations)
        mop        = 1 - ( (count**(1/alpha)) / (iterations**(1/alpha)) )
        population = update_population(population, elite, mu, moa, mop, min_values, max_values, target_function)
        if (population[population[:,-1].argsort()][0,-1] < elite[-1]):
            elite = np.copy(population[population[:,-1].argsort()][0,:]) 
        if (target_value is not None):
            if (elite[-1] <= target_value):
                count = 2* iterations
            else:
                count = count + 1
        else:
            count = count + 1
    return elite

############################################################################

In [ ]:
min_qubits = 3
max_qubits=10
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={"shots": 5120},
            )
    alo = arithmetic_optimization_algorithm(size = 500, min_values = [-100] * dimension, max_values = [100] * dimension, iterations = 1500, alpha = 0.5, mu = 5, 
                                            target_function = evaluate_expectation, verbose = True, start_init = None, target_value = None, qubits = k)

ansatz_num_parameters= 12
Iteration =  0  f(x) =  -1.46875
Iteration =  1  f(x) =  -1.46875
Iteration =  2  f(x) =  -1.46875
Iteration =  3  f(x) =  -1.46875
Iteration =  4  f(x) =  -1.46875
Iteration =  5  f(x) =  -1.46875
Iteration =  6  f(x) =  -1.46875
Iteration =  7  f(x) =  -1.46875


KeyboardInterrupt: 

In [16]:
def rastrigin(x):
    A = 10
    return A * len(x) + sum([(xi**2 - A * np.cos(2 * np.pi * xi)) for xi in x])

alo = arithmetic_optimization_algorithm(size = 500, min_values = [-100] * dimension, max_values = [100] * dimension, iterations = 1500, alpha = 0.5, mu = 5, 
                                            target_function = evaluate_expectation, verbose = True, start_init = None, target_value = None, qubits = k)

Iteration =  0  f(x) =  -1.8125
Iteration =  1  f(x) =  -1.8125
Iteration =  2  f(x) =  -1.8125
Iteration =  3  f(x) =  -1.8125
Iteration =  4  f(x) =  -1.8125
Iteration =  5  f(x) =  -1.8125
Iteration =  6  f(x) =  -1.8125
Iteration =  7  f(x) =  -1.8125
Iteration =  8  f(x) =  -1.8125
Iteration =  9  f(x) =  -1.8125
Iteration =  10  f(x) =  -1.8125
Iteration =  11  f(x) =  -1.8125
Iteration =  12  f(x) =  -1.8125


KeyboardInterrupt: 

In [26]:
# Required Libraries
import numpy as np

# Function: Rastrigin
def rastrigin_function(X):
    A = 10
    return A * len(X) + sum([(x**2 - A * np.cos(2 * np.pi * x)) for x in X])

# Function: Initialize Variables
def initial_variables(size, min_values, max_values, target_function, start_init=None):
    dim = len(min_values)
    if start_init is not None:
        start_init = np.atleast_2d(start_init)
        n_rows = size - start_init.shape[0]
        if n_rows > 0:
            rows = np.random.uniform(min_values, max_values, (n_rows, dim))
            start_init = np.vstack((start_init[:, :dim], rows))
        else:
            start_init = start_init[:size, :dim]
        fitness_values = target_function(start_init) if hasattr(target_function, 'vectorized') else np.apply_along_axis(target_function, 1, start_init)
        population = np.hstack((start_init, fitness_values[:, np.newaxis] if not hasattr(target_function, 'vectorized') else fitness_values))
    else:
        population = np.random.uniform(min_values, max_values, (size, dim))
        fitness_values = target_function(population) if hasattr(target_function, 'vectorized') else np.apply_along_axis(target_function, 1, population)
        population = np.hstack((population, fitness_values[:, np.newaxis] if not hasattr(target_function, 'vectorized') else fitness_values))
    return population

# Function: Update Population
def update_population(population, elite, mu, moa, mop, min_values, max_values, target_function, func_eval_count):
    e = 2.2204e-16
    dim = len(min_values)
    p = np.copy(population)
    r1 = np.random.rand(population.shape[0], dim)
    r2 = np.random.rand(population.shape[0], dim)
    r3 = np.random.rand(population.shape[0], dim)
    update_1 = np.where(r1 > moa, elite[:-1] / (mop + e) * ((max_values - min_values) * mu + min_values), elite[:-1])
    update_2 = np.where(r2 <= 0.5, update_1 * mop, update_1 - mop)
    update_3 = np.where(r3 > 0.5, update_2 - ((max_values - min_values) * mu + min_values), update_2 + ((max_values - min_values) * mu + min_values))
    up_pos = np.clip(update_3, min_values, max_values)
    for i in range(population.shape[0]):
        new_fitness = target_function(up_pos[i, :])
        func_eval_count += 1
        if new_fitness < population[i, -1]:
            p[i, :-1] = up_pos[i, :]
            p[i, -1] = new_fitness
    return p, func_eval_count

# Function: AOA
def arithmetic_optimization_algorithm(size=250, min_values=[-5, -5], max_values=[5, 5], iterations=1500, alpha=0.5, mu=5, target_function=rastrigin_function, verbose=True, start_init=None, target_value=None, qubits=3):
    population = initial_variables(size, min_values, max_values, target_function, start_init)
    elite = np.copy(population[population[:, -1].argsort()][0, :])
    min_values = np.array(min_values)
    max_values = np.array(max_values)
    count = 0
    func_eval_count = size  # Initial evaluations
    while count <= iterations and elite[-1] - 1 + qubits > 1e-1:
        if verbose:
            print('Iteration = ', count, ' f(x) = ', elite[-1], ' Evaluations = ', func_eval_count)
        moa = 0.2 + count * ((1 - 0.2) / iterations)
        mop = 1 - ((count ** (1 / alpha)) / (iterations ** (1 / alpha)))
        population, func_eval_count = update_population(population, elite, mu, moa, mop, min_values, max_values, target_function, func_eval_count)
        if population[population[:, -1].argsort()][0, -1] < elite[-1]:
            elite = np.copy(population[population[:, -1].argsort()][0, :])
        if target_value is not None:
            if elite[-1] <= target_value:
                count = 2 * iterations
            else:
                count += 1
        else:
            count += 1
    return elite, func_eval_count

# Example usage with Rastrigin function
best_solution, total_evaluations = arithmetic_optimization_algorithm(size=100, min_values=[-10, -10], max_values=[10, 10], iterations=200, target_function=rastrigin_function, verbose=True)
print("Best solution:", best_solution)
print("Total function evaluations:", total_evaluations)


Iteration =  0  f(x) =  10.59608169319148  Evaluations =  100
Iteration =  1  f(x) =  10.59608169319148  Evaluations =  200
Iteration =  2  f(x) =  10.59608169319148  Evaluations =  300
Iteration =  3  f(x) =  10.59608169319148  Evaluations =  400
Iteration =  4  f(x) =  10.59608169319148  Evaluations =  500
Iteration =  5  f(x) =  10.59608169319148  Evaluations =  600
Iteration =  6  f(x) =  10.59608169319148  Evaluations =  700
Iteration =  7  f(x) =  10.59608169319148  Evaluations =  800
Iteration =  8  f(x) =  10.59608169319148  Evaluations =  900
Iteration =  9  f(x) =  10.59608169319148  Evaluations =  1000
Iteration =  10  f(x) =  10.59608169319148  Evaluations =  1100
Iteration =  11  f(x) =  10.59608169319148  Evaluations =  1200
Iteration =  12  f(x) =  10.59608169319148  Evaluations =  1300
Iteration =  13  f(x) =  10.59608169319148  Evaluations =  1400
Iteration =  14  f(x) =  10.59608169319148  Evaluations =  1500
Iteration =  15  f(x) =  10.59608169319148  Evaluations =  